In [1]:
from fastmcp import Client

from pydantic import BaseModel, Field

from qdrant_client import QdrantClient
from qdrant_client.models import Prefetch, Filter, FieldCondition, MatchText, FusionQuery

from langsmith import traceable, get_current_run_tree

from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode

from langchain_core.messages import AIMessage, ToolMessage, convert_to_openai_messages

from jinja2 import Template
from typing import Literal, Dict, Any, Annotated, List, Optional
from IPython.display import Image, display
from operator import add
from openai import OpenAI

from utils.utils import format_ai_message

import openai

import random
import ast
import inspect
import instructor
import json

/Users/wilfriedtcheumaha/Code/yelp-assistant/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Listing available tools in MCP servers on localhost:8001/mcp and localhost:8002/mcp

In [9]:
client = Client("http://localhost:8001/mcp")

In [10]:
async with client:
    tools = await client.list_tools()

In [11]:
tools

[Tool(name='get_formatted_context', title=None, description='Get the top k context, each representing a restaurant for a given query.', inputSchema={'additionalProperties': False, 'properties': {'query': {'type': 'string', 'description': 'The query to get the top k context for'}, 'top_k': {'default': 5, 'type': 'integer', 'description': 'The number of context chunks to retrieve, works best with 5 or more'}}, 'required': ['query'], 'type': 'object'}, outputSchema={'properties': {'result': {'type': 'string'}}, 'required': ['result'], 'type': 'object', 'x-fastmcp-wrap-result': True}, icons=None, annotations=None, meta={'fastmcp': {'tags': []}}, execution=None)]

In [12]:
print("======NAME=======")
print(tools[0].name)
print("======DESCRIPTION=======")
print(tools[0].description)
print("======INPUT SCHEMA=======")
print(tools[0].inputSchema)


======NAME=======
get_formatted_context
======DESCRIPTION=======
Get the top k context, each representing a restaurant for a given query.
======INPUT SCHEMA=======
{'additionalProperties': False, 'properties': {'query': {'type': 'string', 'description': 'The query to get the top k context for'}, 'top_k': {'default': 5, 'type': 'integer', 'description': 'The number of context chunks to retrieve, works best with 5 or more'}}, 'required': ['query'], 'type': 'object'}


In [13]:
client=Client("http://localhost:8002/mcp")

In [14]:
async with client:
    tools = await client.list_tools()
tools
print("======NAME=======")
print(tools[0].name)
print("======DESCRIPTION=======")
print(tools[0].description)
print("======INPUT SCHEMA=======")

======NAME=======
get_formatted_reviews_context
======DESCRIPTION=======
Get the top k reviews context for a given query and business ids.
======INPUT SCHEMA=======


### Execute a tool on one of the running MCP servers

In [ ]:
client = Client("http://localhost:8001/mcp")

async with client:

    result = await client.call_tool("get_formatted_context", {"query": "best restaurants in new orleans?", "top_k": 5})

In [16]:
result

CallToolResult(content=[TextContent(type='text', text="-ID: a1u9Bxrq_fZxl2pgqQUcJA, Name: The Governor, Rating: 4.5, Review Count: 330, State: LA, City: New Orleans, Similarity Score: 0.82356983\n-ID: oBNrLz4EDhiscSlbOl8uAw, Name: Ruby Slipper - New Orleans, Rating: 4.5, Review Count: 5193, State: LA, City: New Orleans, Similarity Score: 0.8124\n-ID: -psZLTe6IJTQUB-bZF7Zyg, Name: Little Korea BBQ, Rating: 4.0, Review Count: 423, State: LA, City: New Orleans, Similarity Score: 0.8045629\n-ID: TUTQeLjq1UbkR5r8mOvMqw, Name: Tito's Ceviche & Pisco, Rating: 4.5, Review Count: 213, State: LA, City: New Orleans, Similarity Score: 0.7906887\n-ID: l5cKnwfmnaG4QRZl7ZyNuA, Name: Ancora, Rating: 4.0, Review Count: 224, State: LA, City: New Orleans, Similarity Score: 0.7821981\n", annotations=None, meta=None)], structured_content={'result': "-ID: a1u9Bxrq_fZxl2pgqQUcJA, Name: The Governor, Rating: 4.5, Review Count: 330, State: LA, City: New Orleans, Similarity Score: 0.82356983\n-ID: oBNrLz4EDhisc

In [18]:
print(result.content[0].text)

-ID: a1u9Bxrq_fZxl2pgqQUcJA, Name: The Governor, Rating: 4.5, Review Count: 330, State: LA, City: New Orleans, Similarity Score: 0.82356983
-ID: oBNrLz4EDhiscSlbOl8uAw, Name: Ruby Slipper - New Orleans, Rating: 4.5, Review Count: 5193, State: LA, City: New Orleans, Similarity Score: 0.8124
-ID: -psZLTe6IJTQUB-bZF7Zyg, Name: Little Korea BBQ, Rating: 4.0, Review Count: 423, State: LA, City: New Orleans, Similarity Score: 0.8045629
-ID: TUTQeLjq1UbkR5r8mOvMqw, Name: Tito's Ceviche & Pisco, Rating: 4.5, Review Count: 213, State: LA, City: New Orleans, Similarity Score: 0.7906887
-ID: l5cKnwfmnaG4QRZl7ZyNuA, Name: Ancora, Rating: 4.0, Review Count: 224, State: LA, City: New Orleans, Similarity Score: 0.7821981



### Function to extract tools definitions of all available tools in provider MCP servers

In [24]:
async def get_tool_descriptions_from_mcp_servers(mcp_servers: list[str]) -> list[dict]:
    tool_descriptions = []
    for server in mcp_servers:
        client = Client(server)
        async with client:
            tools = await client.list_tools()
            for tool in tools:
                desc = tool.description or ""
                summary = desc.split("\n\n")[0].strip()

                returns_block = ""
                if "Returns:" in desc:
                    returns_block = desc.split("Returns:", 1)[1].strip()

                # FastMCP already populates per-param descriptions in inputSchema,
                # so we don't need to re-parse the docstring ourselves.
                properties = dict(tool.inputSchema.get("properties", {}))

                tool_descriptions.append({
                    "name": tool.name,
                    "description": summary,
                    "parameters": {"type": "object", "properties": properties},
                    "required": tool.inputSchema.get("required", []),
                    "returns": {"type": "string", "description": returns_block},
                    "server": server,
                })
    return tool_descriptions

In [25]:
mcp_servers = ["http://localhost:8001/mcp", "http://localhost:8002/mcp"]

In [26]:
tool_descriptions = await get_tool_descriptions_from_mcp_servers(mcp_servers)

In [27]:
tool_descriptions

[{'name': 'get_formatted_context',
  'description': 'Get the top k context, each representing a restaurant for a given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'The query to get the top k context for'},
    'top_k': {'default': 5,
     'type': 'integer',
     'description': 'The number of context chunks to retrieve, works best with 5 or more'}}},
  'required': ['query'],
  'returns': {'type': 'string', 'description': ''},
  'server': 'http://localhost:8001/mcp'},
 {'name': 'get_formatted_reviews_context',
  'description': 'Get the top k reviews context for a given query and business ids.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'The query to get the top k reviews context for'},
    'business_ids': {'items': {'type': 'string'},
     'type': 'array',
     'description': 'The list of business ids to get the reviews context for'},
    'k': {'default': 15,
     

### Agent integration with tools exposed via MCP servers